# Datasets and Dataloaders

In [5]:
from run_easg import EASGData
from dataset import myEASGDataset
from pathlib import Path
from torch_geometric.loader import DataLoader

ann_path = 'annts_in_new_format/'
with open(ann_path + 'verbs.txt') as f:
    verbs = [l.strip() for l in f.readlines()]

with open(ann_path + 'objects.txt') as f:
    objs = [l.strip() for l in f.readlines()]

with open(ann_path + 'relationships.txt') as f:
    rels = [l.strip() for l in f.readlines()]

path_annts = Path("annts_in_new_format")
path_data = Path('data')

train_original = EASGData(path_annts, path_data, 'train', verbs, objs, rels)
validation_original = EASGData(path_annts, path_data, 'val', verbs, objs, rels)

train_dataset = myEASGDataset(train_original)
validation_dataset = myEASGDataset(validation_original)

batch_size = 1
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

In [14]:
train_dataset[0].x.size(1)

2304

# Training the model

In [6]:
import torch
from torch import cuda
from models import EdgeClassifier
from utils import set_wandb_config
import models
import wandb
import torch.nn as nn
from torch.optim import Adam
import torch.optim.lr_scheduler as lr_scheduler

import importlib, models
importlib.reload(models)

obj_dim = 1024                  # original object dimension
verb_dim = 2304                 # original verb dimension
hidden_projection_dim = 1024    # hidden dimension of linear projection 
projection_dim = 512            # final dimension after linear projection
hidden_dim = 256              # hidden dim for gnn network
output_dim = 128                # output dim for gnn network
num_rels = 13                   # number of possible relationships (possible classes)
starting_lr = 0.1
min_lr = 0.0005
num_epochs = 100
device = 'cuda' if cuda.is_available() else print('CUDA NOT AVAILABLE')
# device = 'cpu'

In [7]:
edge_classifier = EdgeClassifier(obj_dim, verb_dim, projection_dim, hidden_dim, output_dim, hidden_projection_dim, num_rels, device)
optimizer = Adam(edge_classifier.parameters(), lr=starting_lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1, min_lr=min_lr)
criterion = nn.BCEWithLogitsLoss()
config = set_wandb_config(num_epochs, hidden_projection_dim, projection_dim, hidden_dim, output_dim, batch_size)

In [4]:
def train(train_loader, model, optimizer, scheduler, criterion, config, num_epochs):
    model = model.to(device)
    wandb.init(project = 'easg_classification', config = config)
    loss_tracker = []
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        counter = 0
        for batch in train_loader:
            counter +=1
            batch = batch.to(device)        
            optimizer.zero_grad()
            output = model(batch.x, batch.edge_index)
            
            loss = criterion(output, batch.y)
            loss_tracker.append(loss.item())
            loss.backward()
            optimizer.step()
            wandb.log({"loss": loss})
            
            total_loss += loss.item() * batch.num_graphs
        scheduler.step()
        current_lr = scheduler.get_lr()[0]
        wandb.log({"current_lr": current_lr})
        
        # Print average loss for the epoch
        average_loss = total_loss / len(train_dataset)
        if num_epochs >= 20:
            if epoch%10 == 0:
                print(f'Epoch {epoch+1}, Loss: {average_loss:.4f}')
        else:
            print(f'Epoch {epoch+1}, Loss: {average_loss:.4f}')
        
    torch.save(model.state_dict(), f'my_trained_models/edge_classifier{num_epochs}-{batch_size}-_mean_pd={projection_dim}_hd={hidden_dim}_outd={output_dim}.pth')
    wandb.finish()

SyntaxError: invalid syntax (1335522925.py, line 24)

In [9]:
train(train_loader, edge_classifier, optimizer, criterion, config, num_epochs)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


wandb: Currently logged in as: scoleri-mr. Use `wandb login --relogin` to force relogin


Epoch 1, Loss: 0.0761


RuntimeError: Parent directory my_trained_models does not exist.